In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [2]:
# Load environment variables and initialize the language model
load_dotenv() 
llm = ChatOpenAI(model="gpt-4o-mini")

In [3]:
# Define the state structure for the order workflow
class OrderState(TypedDict):
    order_id: str
    item: str
    quantity: int
    in_stock: bool
    status: str
    customer_message: str

In [4]:
# Define node functions
def validate_order(state: OrderState):
    print("Validating order...")
    return {}

def check_inventory(state: OrderState):
    print("Checking inventory...")

    if state["quantity"] <= 2:
        return {"in_stock": True}
    else:
        return {"in_stock": False}

def fulfill_order(state: OrderState):
    print("Deciding fulfillment...")

    if state["in_stock"]:
        return {"status": "ORDER_CONFIRMED"}
    else:
        return {"status": "OUT_OF_STOCK"}
    
def generate_message(state: OrderState):
    print("Generating customer message ")

    prompt = f"""
    You are an e-commerce assistant.
    Order ID: {state['order_id']}
    Item: {state['item']}
    Status: {state['status']}
    Write a short, friendly message to the customer.
    """
    response = llm.invoke(prompt).content
    return {"customer_message": response}


In [5]:
# Build the graph
graph_builder = StateGraph(OrderState)

graph_builder.add_node("validate_order", validate_order)
graph_builder.add_node("check_inventory", check_inventory)
graph_builder.add_node("fulfill_order", fulfill_order)
graph_builder.add_node("generate_message", generate_message)

graph_builder.add_edge(START, "validate_order")
graph_builder.add_edge("validate_order", "check_inventory")
graph_builder.add_edge("check_inventory", "fulfill_order")
graph_builder.add_edge("fulfill_order", "generate_message")
graph_builder.add_edge("generate_message", END)

In [6]:
# Configure SQLite-based checkpoint persistence and compile the graph
conn = sqlite3.connect("order-checkpoints.sqlite", check_same_thread=False)
checkpointer = SqliteSaver(conn)
graph = graph_builder.compile(checkpointer=checkpointer)

In [7]:
# Execute the graph with an initial order input
config = {"configurable": {"thread_id": "order-001"}}

graph.invoke(
    {
        "order_id": "ORD-101",
        "item": "Wireless Mouse",
        "quantity": 5
    },
    config=config
)


Validating order...
Checking inventory...
Deciding fulfillment...
Generating customer message 


{'order_id': 'ORD-101',
 'item': 'Wireless Mouse',
 'quantity': 5,
 'in_stock': False,
 'status': 'OUT_OF_STOCK',
 'customer_message': "Subject: Update on Your Order - Wireless Mouse\n\nDear Customer,\n\nThank you for your recent order (Order ID: ORD-101) for the Wireless Mouse. We wanted to let you know that this item is currently out of stock. We apologize for the inconvenience and appreciate your patience as we work to replenish our inventory.\n\nIf you would like to explore alternative products or have any questions, please feel free to reach out. We're here to help!\n\nThank you for your understanding.\n\nBest regards,  \n[Your Name]  \nCustomer Support Team"}

In [8]:
# Retrieve and inspect all saved checkpoints for the thread
for checkpoint in graph.get_state_history({"configurable": {"thread_id": "order-001"}}):
    print(checkpoint.config["configurable"]["checkpoint_id"])
    print(checkpoint.next)
    print()


1f0efff7-3a9c-6ea0-8004-0e06737fe5df
()

1f0efff7-1d29-6771-8003-cff95d4cb804
('generate_message',)

1f0efff7-1d27-607b-8002-ea16ad3140e0
('fulfill_order',)

1f0efff7-1d24-698a-8001-67a775e1b9ac
('check_inventory',)

1f0efff7-1d22-62a0-8000-ded2c6af0597
('validate_order',)

1f0efff7-1d1b-6125-bfff-f2b2b8863139
('__start__',)



In [9]:
# Resume graph execution from a specific checkpoint
graph.invoke(None, config={"configurable": {"thread_id": "order-001", "checkpoint_id": "1f0efff7-1d27-607b-8002-ea16ad3140e0"}})


Deciding fulfillment...
Generating customer message 


{'order_id': 'ORD-101',
 'item': 'Wireless Mouse',
 'quantity': 5,
 'in_stock': False,
 'status': 'OUT_OF_STOCK',
 'customer_message': "Subject: Update on Your Order - ORD-101\n\nHi there!\n\nThank you for your recent order of the Wireless Mouse. We wanted to let you know that, unfortunately, this item is currently out of stock. We understand how important it is for you to receive your order, and we're sorry for any inconvenience this may cause.\n\nWe’re working hard to restock it as quickly as possible and will keep you updated on its availability. If you have any questions or would like assistance with an alternative product, please don’t hesitate to reach out.\n\nThank you for your understanding!\n\nBest regards,  \n[Your Name]  \nCustomer Support Team"}

In [10]:
# Inspect checkpoint history after resuming execution
for checkpoint in graph.get_state_history({"configurable": {"thread_id": "order-001"}}):
    print(checkpoint.config["configurable"]["checkpoint_id"])
    print(checkpoint.next)
    print()

1f0efffb-b667-6674-8004-13ecfe14bc2a
()

1f0efffb-97a3-6b70-8003-b08a9c1bb54d
('generate_message',)

1f0efff7-3a9c-6ea0-8004-0e06737fe5df
()

1f0efff7-1d29-6771-8003-cff95d4cb804
('generate_message',)

1f0efff7-1d27-607b-8002-ea16ad3140e0
('fulfill_order',)

1f0efff7-1d24-698a-8001-67a775e1b9ac
('check_inventory',)

1f0efff7-1d22-62a0-8000-ded2c6af0597
('validate_order',)

1f0efff7-1d1b-6125-bfff-f2b2b8863139
('__start__',)



In [11]:
# Update the graph state at a specific checkpoint
new_config = graph.update_state(
    values={"in_stock": True},
    config={
        "configurable": {
            "thread_id": "order-001",
            "checkpoint_id": "1f0efff7-1d27-607b-8002-ea16ad3140e0",
            "checkpoint_ns": ""
        }
    }
)


In [12]:
# Inspect checkpoint history and state values after updating
for checkpoint in graph.get_state_history({"configurable": {"thread_id": "order-001"}}):
    print(checkpoint.config["configurable"]["checkpoint_id"])
    print(checkpoint.next)
    print(checkpoint.values)
    print()

1f0f0002-30e2-67a4-8003-56eea9adca31
('fulfill_order',)
{'order_id': 'ORD-101', 'item': 'Wireless Mouse', 'quantity': 5, 'in_stock': True}

1f0efffb-b667-6674-8004-13ecfe14bc2a
()
{'order_id': 'ORD-101', 'item': 'Wireless Mouse', 'quantity': 5, 'in_stock': False, 'status': 'OUT_OF_STOCK', 'customer_message': "Subject: Update on Your Order - ORD-101\n\nHi there!\n\nThank you for your recent order of the Wireless Mouse. We wanted to let you know that, unfortunately, this item is currently out of stock. We understand how important it is for you to receive your order, and we're sorry for any inconvenience this may cause.\n\nWe’re working hard to restock it as quickly as possible and will keep you updated on its availability. If you have any questions or would like assistance with an alternative product, please don’t hesitate to reach out.\n\nThank you for your understanding!\n\nBest regards,  \n[Your Name]  \nCustomer Support Team"}

1f0efffb-97a3-6b70-8003-b08a9c1bb54d
('generate_message'

In [13]:
# Continue execution using the updated state configuration
graph.invoke(None, new_config)

Deciding fulfillment...
Generating customer message 


{'order_id': 'ORD-101',
 'item': 'Wireless Mouse',
 'quantity': 5,
 'in_stock': True,
 'status': 'ORDER_CONFIRMED',
 'customer_message': "Subject: Your Order is Confirmed!\n\nHi there!\n\nWe're excited to let you know that your order for the Wireless Mouse (Order ID: ORD-101) has been confirmed! 🎉\n\nThank you for shopping with us. If you have any questions or need assistance, feel free to reach out. We can’t wait for you to enjoy your new purchase!\n\nBest regards,  \n[Your E-commerce Team]"}

In [14]:
# Final checkpoint history after completing resumed execution
for checkpoint in graph.get_state_history({"configurable": {"thread_id": "order-001"}}):
    print(checkpoint.config["configurable"]["checkpoint_id"])
    print(checkpoint.next)
    print()  

1f0f0007-473b-6349-8005-f0a1d96350e5
()

1f0f0007-2b07-6c6b-8004-5edcd7d743e1
('generate_message',)

1f0f0002-30e2-67a4-8003-56eea9adca31
('fulfill_order',)

1f0efffb-b667-6674-8004-13ecfe14bc2a
()

1f0efffb-97a3-6b70-8003-b08a9c1bb54d
('generate_message',)

1f0efff7-3a9c-6ea0-8004-0e06737fe5df
()

1f0efff7-1d29-6771-8003-cff95d4cb804
('generate_message',)

1f0efff7-1d27-607b-8002-ea16ad3140e0
('fulfill_order',)

1f0efff7-1d24-698a-8001-67a775e1b9ac
('check_inventory',)

1f0efff7-1d22-62a0-8000-ded2c6af0597
('validate_order',)

1f0efff7-1d1b-6125-bfff-f2b2b8863139
('__start__',)

